In [1]:
# LandownerAntelope
import tabula
import pandas as pd
import requests
import io
import numpy as np
from publictrust.wildlife import parse_pdf_to_dataframe, create_interactive_map
import geopandas as gpd
from pathlib import Path



In [2]:

landownertags2025_url = "https://wgfd.wyo.gov/media/32708/download?inline" # 2025 Resident Landowner Deer Tag Quotas PDF

# Get the DataFrame
landowner_tags = parse_pdf_to_dataframe(landownertags2025_url)

if not landowner_tags.empty:
    # Determine percentage of tags allocated to the landowner
    landowner_tags['pct_landowner'] = landowner_tags['Issued']/landowner_tags['Quota'] * 100
    ls_tag_dist = []
    for i, row in landowner_tags.iterrows():
        ls_tag_dist.append(f"{row['Issued']} of {row['Quota']}")
    landowner_tags['landowner_tags_per_total'] = ls_tag_dist
else:
    print("PROBLEM WITH READING DATA FROM THE PDF FILE. CANNOT PROCEED!")

Successfully parsed PDF and created DataFrame.


In [6]:
# Review summaries of landowner & resident quota data used in the maps:
print(landowner_tags)

   Hunt_Area Type           Description  Quota  Issued  PP  Applicants  \
0          8    3  ANY WHITE-TAILED DEE     40       0   0           0   
1         10    1  ANTLERED MULE DEER O     60      13   0           0   
2         10    3  ANY WHITE-TAILED DEE     20       4   0           0   
3         11    3  ANY WHITE-TAILED DEE    160       0   0           0   
4         15    3  ANY WHITE-TAILED DEE    400       3   0           0   
..       ...  ...                   ...    ...     ...  ..         ...   
29       164    3  ANY WHITE-TAILED DEE     80       0   0           0   
30       165    1  ANTLERED MULE DEER O     40       0   0           0   
31       165    3  ANY WHITE-TAILED DEE     80       5   0           0   
32       171    3  ANY WHITE-TAILED DEE    120       0   0           0   
33       GEN  NaN               GENERAL   9999       0   0           0   

    pct_landowner landowner_tags_per_total  
0        0.000000                  0 of 40  
1       21.666667    

In [3]:
# Read in the Wyoming G&F Elk Hunt Areas Map
gdf_ha = gpd.read_file("~/Documents/personal/BHA/data_in/gpkg/DeerHuntAreas_-2253109999098330244.gpkg")
gdf_ha['HUNTAREA'] = gdf_ha['HUNTAREA'].astype(str).str.replace(".0","")




In [4]:
# Combine map data with the landowner tags data
gdf_ha_cmbo = gdf_ha.merge(landowner_tags,left_on='HUNTAREA', right_on='Hunt_Area')
gdf_cmbo_type1 = gdf_ha_cmbo[gdf_ha_cmbo['Type']=='1'] # Filter to the type 1 tags

In [ ]:
create_interactive_map(gdf=gdf_cmbo_type1, output_html = "~/Documents/personal/BHA/data_out/deer2025_landowner_type1_bha.html", map_title = '2025 Type 1 Resident Landowner Deer Tag Allocations', img_path = "~/Documents/personal/BHA/data_in/BHALOGOBLACK.clear.png")

Reprojecting data to WGS84 (EPSG:4326)...
Adding polygons and hover interactions...
Adding labels...


/Users/guylitt/git/public_trust_wildlife/publictrust/wildlife.py:128: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = gdf.geometry.centroid.y.mean()
/Users/guylitt/git/public_trust_wildlife/publictrust/wildlife.py:129: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = gdf.geometry.centroid.x.mean()


Map successfully generated: /Users/guylitt/Documents/personal/BHA/data_out/deer2025_landowner_type1_bha.html
